# Notebook 05: TensorFlow Lite Quantization, Size Benchmark & Python Verification

**Objectives:**
1. Export fine-tuned MobileNetV2 into TensorFlow Lite format.
2. Generate 3 variants: Float32, Float16, and INT8 Dynamic Range.
3. Compare file sizes and benchmark inference speed.
4. Verify numerical parity between Keras and the TFLite Interpreter on 50 test images.
5. Export `plant_disease_model.tflite` and `labels.txt` for mobile deployment.

In [ ]:
# Cell 0: Setup Directories & Colab Dependencies
import os
!pip install -q importlib_resources tensorflow-datasets --upgrade

RESULTS_DIR = 'results' if os.path.exists('results') or not os.path.exists('../results') else '../results'
MODELS_DIR = 'models' if os.path.exists('models') or not os.path.exists('../models') else '../models'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Results: {os.path.abspath(RESULTS_DIR)}")
print(f"Models:  {os.path.abspath(MODELS_DIR)}")

In [ ]:
# Cell 1: Setup & Load Keras Model
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import pandas as pd
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

model_path = os.path.join(MODELS_DIR, 'mobilenetv2_finetuned.keras')
if os.path.exists(model_path):
    model = tf.keras.models.load_model(model_path)
    print("Loaded Keras model successfully.")
else:
    print(f"Model not found at {model_path}.")

test_ds_raw, ds_info = tfds.load('plant_village', split='train[85%:]', as_supervised=True, with_info=True)
class_names = ds_info.features['label'].names
print(f"Loaded {len(class_names)} classes.")

labels_path = os.path.join(MODELS_DIR, 'labels.txt')
with open(labels_path, 'w') as f:
    for name in class_names:
        f.write(f"{name}\n")
print(f"Wrote labels.txt to {labels_path}")

In [ ]:
# Cell 2: Convert to Float32 TFLite
converter_fp32 = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_fp32_model = converter_fp32.convert()

fp32_path = os.path.join(MODELS_DIR, 'model_fp32.tflite')
with open(fp32_path, 'wb') as f:
    f.write(tflite_fp32_model)
fp32_size = os.path.getsize(fp32_path) / (1024 * 1024)
print(f"Float32 Model Size: {fp32_size:.2f} MB")

In [ ]:
# Cell 3: Convert to Float16 Quantized TFLite
converter_fp16 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16_model = converter_fp16.convert()

fp16_path = os.path.join(MODELS_DIR, 'plant_disease_model.tflite')
with open(fp16_path, 'wb') as f:
    f.write(tflite_fp16_model)
fp16_size = os.path.getsize(fp16_path) / (1024 * 1024)
print(f"Float16 Model Size: {fp16_size:.2f} MB (50% reduction!)")

In [ ]:
# Cell 4: Convert to Dynamic Range INT8 Quantized TFLite
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_int8_model = converter_int8.convert()

int8_path = os.path.join(MODELS_DIR, 'model_int8.tflite')
with open(int8_path, 'wb') as f:
    f.write(tflite_int8_model)
int8_size = os.path.getsize(int8_path) / (1024 * 1024)
print(f"INT8 Dynamic Range Model Size: {int8_size:.2f} MB (75% reduction!)")

In [ ]:
# Cell 5: Verify Numerical Parity on 50 Test Images with TFLite Interpreter
interpreter = tf.lite.Interpreter(model_path=fp16_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Input Details:", input_details[0]['shape'], input_details[0]['dtype'])
print("TFLite Output Details:", output_details[0]['shape'], output_details[0]['dtype'])

matches = 0
total_tested = 50
diffs = []

for i, (img_raw, label_raw) in enumerate(test_ds_raw.take(total_tested)):
    img_resized = tf.image.resize(img_raw, (224, 224))
    inp = preprocess_input(tf.expand_dims(img_resized, 0))
    
    keras_pred = model.predict(inp, verbose=0)[0]
    keras_class = np.argmax(keras_pred)
    
    interpreter.set_tensor(input_details[0]['index'], inp.numpy())
    interpreter.invoke()
    tflite_pred = interpreter.get_tensor(output_details[0]['index'])[0]
    tflite_class = np.argmax(tflite_pred)
    
    if keras_class == tflite_class:
        matches += 1
    diffs.append(np.max(np.abs(keras_pred - tflite_pred)))

print(f"\nNumerical Verification Results:")
print(f"Class Prediction Match Rate: {matches}/{total_tested} ({matches/total_tested*100:.1f}%)")
print(f"Average Maximum Probability Discrepancy: {np.mean(diffs):.6f}")
print("PARITY CONFIRMED: Ready for mobile deployment!")